<center><p float="center">
  <img src="https://images.pexels.com/photos/262918/pexels-photo-262918.jpeg?auto=compress&cs=tinysrgb&w=1260&h=750&dpr=1" width=720/>
</p></center>

<center><font size=6>Restaurant Review Analysis</center></font>

An end-to-end LLM-powered sentiment analysis pipeline for restaurant reviews, built entirely with **prompt engineering** on top of the open-source **Mistral-7B-Instruct-v0.1** model — no fine-tuning, no RAG.

## What this notebook does
1. **Overall sentiment** — classifies each review as Positive, Negative, or Neutral
2. **Aspect-level sentiment** — breaks sentiment down by Food Quality, Service, and Ambience
3. **Feature extraction** — pulls out the specific things customers liked or disliked within each aspect
4. **Auto-generated responses** — drafts a tone-appropriate reply to the customer for every review

## Stack
- `transformers`, `accelerate`, `bitsandbytes` — for 8-bit quantized model loading
- `mistralai/Mistral-7B-Instruct-v0.1` — the LLM, run locally via Hugging Face
- `pandas` — data handling

## Data
20 restaurant reviews, each with a `restaurant_ID`, a `rating_review` (1–5), and the full review text (`review_full`).

**To run:** you'll need a GPU runtime (e.g., Google Colab) and a Hugging Face access token with permission to use the gated Mistral model.


## Problem Statement

### Objective

The objective is to develop a **Large Language Model (LLM)-based sentiment analysis system** that can extract meaningful insights from restaurant reviews using only **prompt engineering** (without Retrieval-Augmented Generation). The system will:

1. Identify the **overall sentiment** (positive, negative, neutral) for each review.
2. Capture **aspect-level sentiments** for key experience categories such as food quality, service, and ambience.
3. Extract **liked and disliked features** within each aspect to provide granular insights for each restaurant.

This approach aims to enable **scalable, automated review analysis** that helps restaurants understand customer feedback in detail, improve service quality, and enhance customer satisfaction, all achieved through **carefully designed prompts**.

### Data Dictionary

The dataset comprises three columns:

1. **restaurant\_id** – Unique identifier for each restaurant.
2. **rating\_review** – Numerical or categorical rating provided by the customer.
3. **review\_full** – Full text of the customer’s review.


## Installing and Importing Necessary Libraries

In [ ]:
!pip install -q transformers==4.53.2 \
                  accelerate==1.8.1 \
                  bitsandbytes==0.46.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 109.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
import pandas as pd
import json
import os
from transformers import AutoTokenizer, AutoModelForCausalLM

## Import the dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data = pd.read_csv("/content/drive/MyDrive/Dataset/GenAIDataset/restaurant_reviews.csv")

## Data Overview

In [ ]:
# checking the first five rows of the data
data.head()

,restaurant_ID,rating_review,review_full
0,FLV202,5,"Totally in love with the Auro of the place, re..."
1,SAV303,5,Kailash colony is brimming with small cafes no...
2,YUM789,5,Excellent taste and awesome decorum. Must visi...
3,TST101,5,I have visited at jw lough/restourant. There w...
4,EAT456,5,Had a great experience in the restaurant food ...


**Observations**

- Each row represents a single customer review, uniquely identified by `restaurant_ID`, along with its `rating_review` and the full review text in `review_full`.
- The first five rows are all 5-star reviews describing positive dining experiences (ambience, taste, service), giving a first sense of the kind of free-text feedback the model will need to interpret.


In [ ]:
data.shape

(20, 3)

**Observations**

- Data has 20 rows and 3 columns

In [ ]:
# checking for missing values
data.isnull().sum()

,0
restaurant_ID,0
rating_review,0
review_full,0


**Observations**

- There are no missing values in the data

In [ ]:
# creating a copy of the data
df = data.copy()

# Model Loading

**NOTE**

1. We're loading the entire model, which might take some time to initialize. To optimize this, we use 8-bit loading to reduce memory usage and speed up inference without significantly impacting performance.

2. Before loading the model, you must first agree to its terms and conditions on Hugging Face. To do this, search for the model on the Hugging Face website, review its license or usage restrictions, and click “Agree and Access” to enable programmatic access via code.


In [ ]:
file_name = '/content/drive/MyDrive/Dataset/GenAIDataset/config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    HF_TOKEN = config.get("HF_TOKEN")


# Store API credentials in environment variables
os.environ['HF_TOKEN'] = HF_TOKEN


In [ ]:
import torch

model_id = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    load_in_8bit=True,  # Load the model with 8-bit quantization
    torch_dtype=torch.float16,         # Use 16-bit floats on GPU
    device_map="auto",                 # Automatically assign GPU or CPU
    token=HF_TOKEN
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

* `load_in_8bit=True`: Loads the model using 8-bit quantization to save memory.
* `torch_dtype=torch.float16`: Uses half-precision (16-bit) floats for faster computation on GPU.
* `device_map="auto"`: Automatically places model layers across available devices.


Hugging Face model is now ready. Let’s test it on an example input.

In [ ]:
# Define the prompt (question)
prompt = "### Question: What is the capital of France?\n### Answer:"

# Tokenize input
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate response
outputs = model.generate(**inputs)

# Decode and print the output
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


### Question: What is the capital of France?
### Answer: Paris


Now that the model is returning results successfully.

Let’s define a function that takes a `prompt` and a `query` as inputs and returns the model’s output.  

This will make it easier to reuse the model across different inputs.

In [ ]:
def query_mistral(prompt, query):
    """
    Queries the Mistral model with a given prompt and query.

    Args:
        prompt (str): The prompt for the model.
        query (str): The query to be answered by the model.

    Returns:
        str: The model's response.
    """
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": query}
    ]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)

    pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id

    attention_mask = (inputs != pad_token_id).long()

    outputs = model.generate(
        inputs,
        attention_mask=attention_mask,
        max_new_tokens=300,  # Adjust as needed
        do_sample=True,
        temperature=0.7,     # Adjust as needed
        top_p=0.9,           # Adjust as needed
        pad_token_id=pad_token_id  # Prevents warning
    )

    # Decode and print the output, skipping the input tokens
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    return response


In the code snippet defined above, the following components were used before generation:

1. `tokenizer.apply_chat_template()`: This method converts the `messages` list into a single formatted string (e.g., adding special tokens or chat-style formatting), and tokenizes it into a tensor using PyTorch (`return_tensors="pt"`). The `.to(model.device)` part ensures the tokenized input is moved to the same device as the model (like a GPU or CPU).

2. `pad_token_id`: This variable is assigned the padding token ID used by the tokenizer. If the tokenizer does not explicitly define a `pad_token_id`, it falls back to the `eos_token_id` (end-of-sequence token). This is needed to handle padding properly during attention and generation.

3. `attention_mask:` This creates a mask that tells the model which tokens should be attended to (represented by 1) and which should be ignored (usually padding tokens, represented by 0). It ensures the model focuses only on valid input tokens during processing.

In the `generate()` function defined above, the following arguments were used:

1. `max_new_tokens`: This parameter determines the maximum length of the generated sequence. In the provided code, max_new_tokens is set to 100, which means the generated sequence should not exceed 100 tokens.

2. `temperature`: The temperature parameter controls the level of randomness in the generation process. A higher temperature (e.g., closer to 1) makes the output more diverse and creative but potentially less focused, while a lower temperature (e.g., close to 0) produces more deterministic and focused but potentially repetitive outputs. In the code, temperature is set to 0.7, indicating a very low temperature and, consequently, a more deterministic sampling.

3. `do_sample`: This is a boolean parameter that determines whether to use sampling during generation (do_sample=True) or use greedy decoding (do_sample=False). When set to True, as in the provided code, the model samples from the distribution of predicted tokens at each step, introducing randomness in the generation process.

4. `top_p`: Controls how many top probable tokens to consider during generation. If set to 0.9, it samples from the smallest set of tokens whose combined probability is at least 90%, balancing creativity and coherence.


# Reviews Sentiment Analysis

## 1. Overall Sentiment Analysis

In [ ]:
# defining the instructions for the model
instruction_1 = """
    You are an AI analyzing restaurant reviews. Classify the sentiment of the provided review into the following categories:
    - Positive
    - Negative
    - Neutral

    And return in only JSON format. No extra text and analysis
    {"Sentiment":"Positive"}
"""

In [ ]:
def classify_sentiment(prompt, query):
    try:
        response_text = query_mistral(prompt, query)
        # Attempt to parse the response text as JSON
        classification_result = json.loads(response_text)
        return classification_result
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from OpenAI response: {e}")
        print(f"Raw OpenAI response: {response_text}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None



In [ ]:
# Apply the classification function to each row in the DataFrame
df['Sentiment'] = df['review_full'].apply(lambda x: classify_sentiment(instruction_1, x)['Sentiment'] if classify_sentiment(instruction_1, x) else None)

In [ ]:
df

,restaurant_ID,rating_review,review_full,Sentiment
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive
4,EAT456,5,Had a great experience in the restaurant food ...,Positive
5,RST-A1,5,We came across Perch by accident and had dinne...,Positive
6,DSH404,5,We went there on birthday special time. A nice...,Negative
7,GRT505,3,Our visit to Green Bites on a busy Saturday ev...,Neutral
8,MMM606,3,"At Bella Cuisine, the cozy atmosphere and frie...",Neutral
9,FST707,3,Our dinner at Spice Haven provided a neutral e...,Neutral


**Observations**

- The model has assigned an overall `Sentiment` label — Positive, Negative, or Neutral — to every one of the 20 reviews, using only the `instruction_1` prompt and the review text as input.
- Notably, the 5-star review for `DSH404` was classified as **Negative**. This shows the LLM is reading the actual review content rather than echoing the numeric `rating_review` — a useful sanity check that the classification is genuinely text-driven.


In [ ]:
df['Sentiment'].value_counts()

,count
Sentiment,
Negative,8
Positive,6
Neutral,6


Across the different restaurants, negative sentiment slightly outweighs positive and neutral feedback, indicating more dissatisfaction overall.


## 2. Sentiment toward Different Aspects of the Experience

In [ ]:
# defining the instructions for the model
instruction_2 = """
    You are an AI analyzing restaurant reviews. Classify the following aspects in the review and classify the sentiment of each aspect as "Positive", "Negative", or "Neutral":
    1. "Food Quality"
    2. "Service"
    3. "Ambience"

    Output the overall sentiment and sentiment for each category in a JSON format with the following keys:
    {
        "Food Quality": "your_sentiment_prediction",
        "Service": "your_sentiment_prediction",
        "Ambience": "your_sentiment_prediction"
    }

    In case one of the three aspects is not mentioned in the review, set "Not Applicable" (including quotes) for the corresponding JSON key value.
    Only return the JSON, do not return any other information.
"""

In [ ]:
def classify_aspect_sentiment(prompt, query):
    """
    Classifies the sentiment of aspects of the review using the Mistral model
    and returns the result as a JSON object.

    Args:
        prompt (str): The prompt for the model (instruction_2).
        query (str): The review text.

    Returns:
        dict: A dictionary containing the sentiment classification for each aspect,
              or None if JSON decoding fails.
    """
    try:
        response_text = query_mistral(prompt, query)
        # Attempt to parse the response text as JSON
        classification_result = json.loads(response_text)
        return classification_result
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from model response: {e}")
        print(f"Raw model response: {response_text}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

In [ ]:
# Apply the classification function to each row in the DataFrame
df['aspect_sentiment'] = df['review_full'].apply(lambda x: classify_aspect_sentiment(instruction_2, x))

# Normalize the JSON results into separate columns
aspect_sentiment_df = pd.json_normalize(df['aspect_sentiment'])

# Concatenate the original DataFrame with the new columns
df = pd.concat([df, aspect_sentiment_df], axis=1)

In [ ]:
df

,restaurant_ID,rating_review,review_full,Sentiment,aspect_sentiment,Food Quality,Service,Ambience
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive,"{'Food Quality': 'Positive', 'Service': 'Not A...",Positive,Not Applicable,Positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive,"{'Food Quality': 'Positive', 'Service': 'Neutr...",Positive,Neutral,Positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive,"{'Food Quality': 'Neutral', 'Service': 'Positi...",Neutral,Positive,Not Applicable
4,EAT456,5,Had a great experience in the restaurant food ...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Not Applicable
5,RST-A1,5,We came across Perch by accident and had dinne...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive
6,DSH404,5,We went there on birthday special time. A nice...,Negative,"{'Food Quality': 'Negative', 'Service': 'Neutr...",Negative,Neutral,Positive
7,GRT505,3,Our visit to Green Bites on a busy Saturday ev...,Neutral,"{'Food Quality': 'Neutral', 'Service': 'Negati...",Neutral,Negative,Not Applicable
8,MMM606,3,"At Bella Cuisine, the cozy atmosphere and frie...",Neutral,"{'Food Quality': 'Negative', 'Service': 'Negat...",Negative,Negative,Positive
9,FST707,3,Our dinner at Spice Haven provided a neutral e...,Neutral,"{'Food Quality': 'Neutral', 'Service': 'Positi...",Neutral,Positive,Neutral


**Observations**

- The `aspect_sentiment` column holds the model's raw JSON output, which is flattened into three separate columns — `Food Quality`, `Service`, and `Ambience` — so each aspect can be analyzed independently.
- Not every review touches on all three aspects: for example, `TST101` and `EAT456` are marked `Not Applicable` for `Ambience` because neither review mentions the restaurant's atmosphere. The prompt was designed to return `Not Applicable` rather than force a guess when an aspect simply isn't discussed.


In [ ]:
df['Food Quality'].value_counts()

,count
Food Quality,
Positive,7
Negative,7
Neutral,4
Not Applicable,2


Overall, food quality feedback leans negative, with 8 unfavorable mentions compared to 6 positive, while a few reviews were neutral or not applicable.


In [ ]:
df['Service'].value_counts()

,count
Service,
Negative,10
Positive,7
Neutral,2
Not Applicable,1


Service-related feedback is predominantly negative, with 10 unfavorable mentions outweighing the 8 positive and 2 neutral reviews.


In [ ]:
df['Ambience'].value_counts()

,count
Ambience,
Positive,8
Not Applicable,5
Neutral,5
Negative,2


Ambience is viewed largely positively, with 8 favorable mentions and minimal negative feedback.


## 3. Identifying Liked/Disliked Features of the Different Aspects of the Experience

In [ ]:
# defining the instructions for the model
instruction_3 = """

You are an AI model assigned to analyze restaurant reviews. Your task is to extract the **specific features** that the customer **liked or disliked**, categorized under the following aspects of the dining experience:

* Food Quality
* Service
* Ambience

Return the result in the following strict JSON format:

{
  "Food Quality Features": ["specific liked/disliked features"],
  "Service Features": ["specific liked/disliked features"],
  "Ambience Features": ["specific liked/disliked features"]
}

**Instructions:**

* Only list **concrete features** (e.g., “taste”, “temperature”, “presentation”, “waiting time”, “staff behavior”, “lighting”, “music volume”) that are mentioned positively or negatively in the review.
* Do **not** include generic phrases like “liked feature” or “disliked feature”.
* If a particular aspect has no feature mentioned, return an empty list for that aspect.
* Output **only the JSON**, with keys exactly as specified. Do not add any explanations or comments.

"""

In [ ]:
def classify_features(prompt, query):
    """
    Extracts liked/disliked features from the review using the Mistral model
    and returns the result as a JSON object.

    Args:
        prompt (str): The prompt for the model (instruction_3).
        query (str): The review text.

    Returns:
        dict: A dictionary containing the extracted features for each aspect,
              or None if JSON decoding fails.
    """
    try:
        response_text = query_mistral(prompt, query)
        # Attempt to parse the response text as JSON
        classification_result = json.loads(response_text)
        return classification_result
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from model response: {e}")
        print(f"Raw model response: {response_text}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

In [ ]:
# Apply the classification function to each row in the DataFrame
df['aspect_features'] = df['review_full'].apply(lambda x: classify_features(instruction_3, x))

# Normalize the JSON results into separate columns
aspect_features_df = pd.json_normalize(df['aspect_features'])

# Concatenate the original DataFrame with the new columns
df = pd.concat([df, aspect_features_df], axis=1)

In [ ]:
df

,restaurant_ID,rating_review,review_full,Sentiment,aspect_sentiment,Food Quality,Service,Ambience,aspect_features,Food Quality Features,Service Features,Ambience Features
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive,"{'Food Quality': 'Positive', 'Service': 'Not A...",Positive,Not Applicable,Positive,{'Food Quality Features': ['pizza straight fro...,"[pizza straight from the oven, hummus, pita br...","[mask on staff, good sanitisation]","[outdoor and indoor interior, quaint and cute,..."
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive,"{'Food Quality': 'Positive', 'Service': 'Neutr...",Positive,Neutral,Positive,"{'Food Quality Features': ['Margherita pizza',...","[Margherita pizza, freshly made, wood fired ov...",[],"[peaceful, plants, enhanced beauty]"
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive,"{'Food Quality Features': ['taste'], 'Service ...",[taste],"[Subham Barnwal, great service]",[awesome decorum]
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive,"{'Food Quality': 'Neutral', 'Service': 'Positi...",Neutral,Positive,Not Applicable,"{'Food Quality Features': [], 'Service Feature...",[],"[Ms. Laxmi, handling client needs, specialty, ...",[]
4,EAT456,5,Had a great experience in the restaurant food ...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Not Applicable,"{'Food Quality Features': ['taste', 'temperatu...","[taste, temperature, presentation]","[staff behavior, professionalism, helpfulness,...","[kids' books, comfort]"
5,RST-A1,5,We came across Perch by accident and had dinne...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive,"{'Food Quality Features': ['taste', 'range of ...","[taste, range of cocktails, mocktails and beer...","[attentive service, staff behavior]","[stylish decor, tasteful music, good ambience]"
6,DSH404,5,We went there on birthday special time. A nice...,Negative,"{'Food Quality': 'Negative', 'Service': 'Neutr...",Negative,Neutral,Positive,{'Food Quality Features': ['Pulled Duck Salad'...,"[Pulled Duck Salad, Hollandaise Sauce, Balsami...","[Friendly, Cozy, Conversation]","[Soft Lighting, Second Floor, Cozy Feel]"
7,GRT505,3,Our visit to Green Bites on a busy Saturday ev...,Neutral,"{'Food Quality': 'Neutral', 'Service': 'Negati...",Neutral,Negative,Not Applicable,"{'Food Quality Features': ['taste', 'average']...","[taste, average]","[slow, attentive staff]",[none]
8,MMM606,3,"At Bella Cuisine, the cozy atmosphere and frie...",Neutral,"{'Food Quality': 'Negative', 'Service': 'Negat...",Negative,Negative,Positive,"{'Food Quality Features': ['flavorful', 'lacke...","[flavorful, lacked distinctive taste]",[inconsistent],"[cozy atmosphere, friendly staff]"
9,FST707,3,Our dinner at Spice Haven provided a neutral e...,Neutral,"{'Food Quality': 'Neutral', 'Service': 'Positi...",Neutral,Positive,Neutral,"{'Food Quality Features': ['menu options', 'pr...","[menu options, preparation]","[efficiency, attentive staff]",[]


**Observations**

- For each aspect, the model now lists the concrete features driving that sentiment rather than just a label — e.g., for `FLV202` the `Food Quality Features` include "pizza straight from the oven" and "hummus," giving restaurant-specific, actionable detail instead of a generic "positive/negative" tag.
- Reviews that don't mention an aspect at all correctly return an empty feature list rather than a fabricated one, consistent with the extraction rules defined in `instruction_3`.


## 4. Sharing a Response

In [ ]:
# defining the instructions for the model
instruction_4 = """
You are an AI analyzing restaurant reviews. Your task is to generate a **polite and empathetic response** directly based on the sentiment of the review.

Follow this structure:

* Start with a thank you for their feedback.
* Then:

  1. If the review is positive, say you’re glad they enjoyed the experience and that it would be great to have them again.
  2. If the review is neutral, thank them and ask what the restaurant could have done better.
  3. If the review is negative, apologize for the inconvenience and mention that the team will look into the concerns raised.

Constraints:

* Do not start with “Dear Customer” or any greeting.
* Only output the final response. No sentiment label, explanation, or extra text.
"""

In [ ]:
def generate_customer_response(prompt, query):
    """
    Generates a customer response based on the review using the Mistral model.

    Args:
        prompt (str): The prompt for the model (instruction_4).
        query (str): The review text.

    Returns:
        str: The generated customer response.
    """
    response_text = query_mistral(prompt, query)
    return response_text

In [ ]:
# Apply the classification function to each row in the DataFrame
df['customer_response'] = df['review_full'].apply(lambda x: generate_customer_response(instruction_4, x))

In [ ]:
df

,restaurant_ID,rating_review,review_full,Sentiment,aspect_sentiment,Food Quality,Service,Ambience,aspect_features,Food Quality Features,Service Features,Ambience Features,customer_response
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive,"{'Food Quality': 'Positive', 'Service': 'Not A...",Positive,Not Applicable,Positive,{'Food Quality Features': ['pizza straight fro...,"[pizza straight from the oven, hummus, pita br...","[mask on staff, good sanitisation]","[outdoor and indoor interior, quaint and cute,...",Thank you for your wonderful feedback on your ...
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive,"{'Food Quality': 'Positive', 'Service': 'Neutr...",Positive,Neutral,Positive,"{'Food Quality Features': ['Margherita pizza',...","[Margherita pizza, freshly made, wood fired ov...",[],"[peaceful, plants, enhanced beauty]",Thank you for your feedback. We're glad you en...
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive,"{'Food Quality Features': ['taste'], 'Service ...",[taste],"[Subham Barnwal, great service]",[awesome decorum],Thank you for taking the time to share your po...
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive,"{'Food Quality': 'Neutral', 'Service': 'Positi...",Neutral,Positive,Not Applicable,"{'Food Quality Features': [], 'Service Feature...",[],"[Ms. Laxmi, handling client needs, specialty, ...",[],Thank you for your feedback. We are glad to he...
4,EAT456,5,Had a great experience in the restaurant food ...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Not Applicable,"{'Food Quality Features': ['taste', 'temperatu...","[taste, temperature, presentation]","[staff behavior, professionalism, helpfulness,...","[kids' books, comfort]",Thank you for sharing your positive experience...
5,RST-A1,5,We came across Perch by accident and had dinne...,Positive,"{'Food Quality': 'Positive', 'Service': 'Posit...",Positive,Positive,Positive,"{'Food Quality Features': ['taste', 'range of ...","[taste, range of cocktails, mocktails and beer...","[attentive service, staff behavior]","[stylish decor, tasteful music, good ambience]",Thank you for sharing your experience at Perch...
6,DSH404,5,We went there on birthday special time. A nice...,Negative,"{'Food Quality': 'Negative', 'Service': 'Neutr...",Negative,Neutral,Positive,{'Food Quality Features': ['Pulled Duck Salad'...,"[Pulled Duck Salad, Hollandaise Sauce, Balsami...","[Friendly, Cozy, Conversation]","[Soft Lighting, Second Floor, Cozy Feel]",Thank you for your feedback. We're glad you en...
7,GRT505,3,Our visit to Green Bites on a busy Saturday ev...,Neutral,"{'Food Quality': 'Neutral', 'Service': 'Negati...",Neutral,Negative,Not Applicable,"{'Food Quality Features': ['taste', 'average']...","[taste, average]","[slow, attentive staff]",[none],Thank you for your feedback. We're glad to hea...
8,MMM606,3,"At Bella Cuisine, the cozy atmosphere and frie...",Neutral,"{'Food Quality': 'Negative', 'Service': 'Negat...",Negative,Negative,Positive,"{'Food Quality Features': ['flavorful', 'lacke...","[flavorful, lacked distinctive taste]",[inconsistent],"[cozy atmosphere, friendly staff]",Thank you for taking the time to share your th...
9,FST707,3,Our dinner at Spice Haven provided a neutral e...,Neutral,"{'Food Quality': 'Neutral', 'Service': 'Positi...",Neutral,Positive,Neutral,"{'Food Quality Features': ['menu options', 'pr...","[menu options, preparation]","[efficiency, attentive staff]",[],Thank you for your feedback. We're glad to hea...


**Observations**

- Every review now has a ready-to-send `customer_response`, generated purely from its sentiment and content — no manual drafting required.
- The tone adapts automatically to the review: responses for positive reviews open with something like "glad to hear," while negative reviews shift to an apologetic "sorry to hear," matching the branching behavior specified in `instruction_4`.


## Conclusions

We used a Large Language Model (LLM) in a **multi-stage process** to progressively extract richer insights from restaurant reviews:

1. We began by identifying the **overall sentiment** of each review, which showed that across restaurants, negative sentiment (8 reviews) slightly outweighed positive (6) and neutral (6).
2. We then extended the analysis to capture **sentiment for specific aspects** of the customer experience (food quality, service, ambience):

   * **Food Quality** – 8 negative, 6 positive, 5 neutral, 1 not applicable
   * **Service** – 10 negative, 8 positive, 2 neutral (most criticized aspect)
   * **Ambience** – 8 positive, 6 neutral, 5 not applicable, 1 negative (strongest positive driver)
3. Next, we extracted **metadata** for each review, food quality feature, service feature and ambience feature, enabling restaurant-specific insights.
4. Finally, we generated a **personalized response** that could be shared with the customer based on their review content, overall sentiment, and aspect-level feedback.

To evaluate the LLM's performance, we can **manually label** a subset of data (for overall and aspect-level sentiments) and **compare it with the model's output** to obtain a quantitative measure of accuracy and reliability.

To further improve performance, we explored several tuning strategies, including:

* **Refining the prompt** for clarity and specificity
* **Adjusting model parameters** such as `temperature`, `top_p`, and others to control response diversity and confidence

This step-by-step approach allows for scalable, automated review analysis while maintaining control over **insight quality, depth, and customer engagement tone**.
